# NB2 — practice (paper Section 3b)

Computes the results of Section 3b: full-batch Adam on data at matched parameter counts, and Levenberg–Marquardt on the same runs. Output goes to `results/nb2/`; figures and tables are made by `figures.ipynb`.
`RUN_MODE`: `'full'` (paper), `'reduced'` or `'quick'`.
Cells 2b, 2c and 2d are robustness checks added after the audit: 2b brackets the learning-rate grid where the near-best plateau touches its edge (`tuning_ext.csv`, `ext_selection.csv`, `eval_ext.csv`, `eval_bracketed.csv`, `summary_ext.csv`, `shift_serial_minus_ancilla_ext.csv`), 2c repeats a pre-specified subset with four times as many Adam steps under a stretched schedule (`budget.csv`, `budget_summary.csv`), 2d with the extra steps at the initial rate and the final decay unchanged (`budget_hold.csv`), and gives success rates for all schedules (`budget_rates.csv`). None changes a primary result file.

Cell 3 (Levenberg–Marquardt on the same runs) covers every configuration of cell 2 when `LM_SCOPE = 'all'` (set in cell 0); it is resumable and skips the configurations already in `attribution.csv`.

In [ ]:
# ── Cell 0: configuration ─────────────────────────────────────────────────
import time, warnings, io, urllib.request
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import jax, jax.numpy as jnp
from phaselock_core import ARMS, Circuit, check_definition, check_gradient, random_target, ls_oracle, adam, lm_svd, plateau_centre, hl_shift
warnings.filterwarnings('ignore'); pd.set_option('display.width', 250); pd.set_option('display.max_columns', 40)
print('JAX', jax.__version__, jax.devices())

RUN_MODE = 'full'                                        # 'full' | 'reduced' | 'quick'
SEED = 3031
TBL = {'serial': 5, 'ancilla': 1, 'parallel2': 2}        # 15 / 15 / 30 trainable parameters per position
ARM_LIST = list(TBL)
MODE = {'full':    dict(E=[8, 12, 16, 24, 32], E_real=[24, 32], runs=100, tune=5, steps=20000, lm=2000, attr_E=[12, 24]),
        'reduced': dict(E=[8, 16, 24],         E_real=[24],     runs=30,  tune=3, steps=20000, lm=2000, attr_E=[24]),
        'quick':   dict(E=[8, 12],             E_real=[8],      runs=3,   tune=2, steps=400,   lm=30,   attr_E=[12])}[RUN_MODE]
E_LIST, E_REAL, N_RUNS, N_TUNE, STEPS, LM_ITERS, ATTR_E = (MODE[k] for k in ('E', 'E_real', 'runs', 'tune', 'steps', 'lm', 'attr_E'))
FAMILIES = {'flat': 0.9, 'bump': 0.8}                    # family -> sup|y| on a dense grid
LRS = [1e-3, 3e-3, 1e-2, 3e-2, 1e-1]
CHUNK = 500 if RUN_MODE != 'quick' else 200
ATTR_ARMS = ['serial', 'ancilla']
LM_SCOPE = 'all'                                         # Levenberg-Marquardt reference (cell 3). 'all': every configuration of cell 2 (synthetic
                                                         # and real, the three architectures); 'subset': synthetic families at E in ATTR_E, ATTR_ARMS
                                                         # (the earlier choice). ATTR_E and ATTR_ARMS still define the subset of cells 2c and 2d.
NOTTEM_URL = 'https://raw.githubusercontent.com/vincentarelbundock/Rdatasets/master/csv/datasets/nottem.csv'
NOTTEM_LOCAL = Path('data/nottem.csv')
SCREENED = {'gasprice': ('https://…/quantreg/gasprice.csv', Path('data/gasprice.csv')),
               'ibmclose': ('https://…/timeSeriesDataSets/ibmclose_ts.csv', Path('data/ibmclose_ts.csv'))}
LRS_REAL = [3e-4] + LRS                                  # screened series only: on Nottingham serial's optimum sat at the grid edge
OUT = Path('results' + ('' if RUN_MODE == 'full' else f'_{RUN_MODE}')) / 'nb2'; OUT.mkdir(parents=True, exist_ok=True)
ARM_ID = {a: i for i, a in enumerate(ARM_LIST)}; FAM_ID = {'flat': 0, 'bump': 1, 'nottem': 2, 'gasprice': 3, 'ibmclose': 4}
X_ALL = np.linspace(-np.pi, np.pi, 300, endpoint=False)
perm = np.random.default_rng(SEED).permutation(300); ITR, ITE = np.sort(perm[:200]), np.sort(perm[200:])
def target_seed(fam, E, t): return SEED + 10**6 * FAM_ID[fam] + 10**4 * E + t          # t < 1000: evaluation, 5000+: tuning
def init_seed(arm, fam, E, t): return SEED + 10**8 * (ARM_ID[arm] + 1) + 10**6 * FAM_ID[fam] + 10**4 * E + t
print('out:', OUT)

In [ ]:
# ── Cell 1: data sets, oracle, checks ─────────────────────────────────────────
def synthetic(fam, E, t):
    y = random_target(fam, E, target_seed(fam, E, t), FAMILIES[fam])(X_ALL); xtr, ytr, xte, yte = X_ALL[ITR], y[ITR], X_ALL[ITE], y[ITE]
    _, L_or, sup = ls_oracle(E, xtr, ytr); assert sup < 1 - 1e-9, f'{fam} E={E} t={t}: oracle infeasible (sup {sup:.4f})'
    return dict(xtr=xtr, ytr=ytr, xte=xte, yte=yte, L_or=L_or, V=float(np.var(ytr)), Vte=float(np.var(yte)))
try:
    raw = pd.read_csv(NOTTEM_LOCAL) if NOTTEM_LOCAL.exists() else pd.read_csv(io.StringIO(urllib.request.urlopen(NOTTEM_URL, timeout=30).read().decode()))
    NOTTEM = raw['value'].to_numpy(float); assert len(NOTTEM) == 240
except Exception as e:
    NOTTEM = None; print('!! Nottingham data not available:', e)
def nottem(E):
    """Whole series (20 years) on [-pi, pi); fixed 180/60 split; centred and scaled so that the degree-E least-squares fit has sup 0.9."""
    x = np.linspace(-np.pi, np.pi, 240, endpoint=False); p = np.random.default_rng(SEED + 1).permutation(240); itr, ite = np.sort(p[:180]), np.sort(p[180:])
    y = NOTTEM - NOTTEM[itr].mean(); _, _, sup = ls_oracle(E, x[itr], y[itr]); y = 0.9 * y / sup
    _, L_or, sup = ls_oracle(E, x[itr], y[itr]); assert abs(sup - 0.9) < 1e-9
    return dict(xtr=x[itr], ytr=y[itr], xte=x[ite], yte=y[ite], L_or=L_or, V=float(np.var(y[itr])), Vte=float(np.var(y[ite])))
SERIES = {}
for fam, (url, local) in SCREENED.items():
    try:
        raw = pd.read_csv(local) if local.exists() else pd.read_csv(io.StringIO(urllib.request.urlopen(url, timeout=30).read().decode()))
        v = raw.sort_values('time')['value'].to_numpy(float)[-300:]; assert len(v) == 300 and np.all(np.isfinite(v))
        k = np.arange(300.0); SERIES[fam] = v - np.polyval(np.polyfit(k, v, 1), k)          # affine trend removed (as in the screen)
    except Exception as e:
        print(f'!! {fam} data not available:', e)
def screened(fam, E):
    """Last 300 points on [-pi, pi), affine trend removed; fixed random 200/100 split; centred on the train mean and
    scaled so that the degree-E least-squares fit has sup 0.9 (the Nottingham convention)."""
    x = np.linspace(-np.pi, np.pi, 300, endpoint=False); p = np.random.default_rng(SEED + FAM_ID[fam]).permutation(300)
    itr, ite = np.sort(p[:200]), np.sort(p[200:])
    y = SERIES[fam] - SERIES[fam][itr].mean(); _, _, sup = ls_oracle(E, x[itr], y[itr]); y = 0.9 * y / sup
    _, L_or, sup = ls_oracle(E, x[itr], y[itr]); assert abs(sup - 0.9) < 1e-9
    return dict(xtr=x[itr], ytr=y[itr], xte=x[ite], yte=y[ite], L_or=L_or, V=float(np.var(y[itr])), Vte=float(np.var(y[ite])))
REAL = {**({'nottem': nottem} if NOTTEM is not None else {}), **{f: (lambda E, f=f: screened(f, E)) for f in SERIES}}
for arm in ARM_LIST: print(f'definition check {arm:9s} (tbl {TBL[arm]}): {check_definition(arm, 8, TBL[arm]):.1e}')
d = synthetic('flat', 8, 0); c = Circuit('serial', 8, TBL['serial']); print('Lie gradient vs finite difference:', check_gradient(c, jnp.asarray(d['xtr']), jnp.asarray(d['ytr']), d['V']))
for fam in FAMILIES:
    for E in E_LIST:
        g = [synthetic(fam, E, t) for t in range(3)]
        print(f'{fam:5s} E={E:2d}: oracle log10(L_or/Var) = {np.median([np.log10(max(x["L_or"] / x["V"], 1e-16)) for x in g]):6.2f} (median of 3)')
def oracle_test_R2(E, d):
    from phaselock_core import trig_basis
    c, _, _ = ls_oracle(E, d['xtr'], d['ytr']); return float(1 - np.mean((trig_basis(d['xte'], E) @ c - d['yte']) ** 2) / d['Vte'])
for fam, load in REAL.items():
    for E in E_REAL:
        d = load(E); print(f'{fam:8s} E={E}: oracle log10(L_or/Var) = {np.log10(d["L_or"] / d["V"]):.2f}, oracle test R2 = {oracle_test_R2(E, d):.3f}')

def run_cell(arm, E, datasets, seeds, lr, mode, steps=None, hold=0):
    """mode: 'adam' full-batch Adam (STEPS steps, or `steps` if given; `hold`: steps at the initial rate before the decay), 'lm' Levenberg-Marquardt. All runs share the training x-grid."""
    c = Circuit(arm, E, TBL[arm]); X = jnp.asarray(datasets[0]['xtr'])
    Y = jnp.asarray(np.array([d['ytr'] for d in datasets])); V = jnp.asarray([d['V'] for d in datasets]); U0 = jnp.stack([c.init(s) for s in seeds])
    if mode == 'lm': H, UF = lm_svd(c, U0, X, Y, V, LM_ITERS)
    else: H, UF = adam(c, U0, X, Y, V, lr, steps or STEPS, chunk=CHUNK, hold=hold)
    ftest = jax.jit(jax.vmap(lambda U: c.f(U, jnp.asarray(datasets[0]['xte']))))(UF)
    out = []
    for r, d in enumerate(datasets):
        Ltr = float(H[r, -1]) * d['V']; out.append(dict(lexc=float(np.log10(max((Ltr - d['L_or']) / d['V'], 1e-16))),
                                                        test_R2=float(1 - np.mean((np.asarray(ftest[r]) - d['yte']) ** 2) / d['Vte'])))
    return out

In [3]:
# ── Cell 2: tuning (separate targets) and evaluation (100 independent runs per cell) ──
PT, PE = OUT / 'tuning.csv', OUT / 'eval.csv'
done = set(map(tuple, pd.read_csv(PE)[['arm', 'family', 'E']].values)) if PE.exists() else set()
LRSTAR = {}
cells = [(fam, E) for fam in FAMILIES for E in E_LIST] + [(fam, E) for fam in REAL for E in E_REAL]
print('cells already evaluated (skipped):', len(done), '| cells to evaluate:', sorted({(a, f, E) for f, E in cells for a in ARM_LIST if not E % ARMS[a]['div']} - done))
for fam, E in cells:
    for arm in ARM_LIST:
        if E % ARMS[arm]['div']: continue
        t0 = time.time()
        grid = LRS_REAL if fam in SCREENED else LRS                       # synthetic and Nottingham keep their grid, so nothing is re-tuned
        if fam in REAL: dt, st = [REAL[fam](E)] * N_TUNE, [init_seed(arm, fam, E, 5000 + j) for j in range(N_TUNE)]
        else: dt, st = [synthetic(fam, E, 5000 + j) for j in range(N_TUNE)], [init_seed(arm, fam, E, 5000 + j) for j in range(N_TUNE)]
        prev = pd.read_csv(PT) if PT.exists() else pd.DataFrame(columns=['arm', 'family', 'E', 'lr', 'q75'])
        scores = {r.lr: r.q75 for r in prev[(prev.arm == arm) & (prev.family == fam) & (prev.E == E)].itertuples()}
        for lr in [l for l in grid if l not in scores]:
            res = run_cell(arm, E, dt, st, lr, 'adam'); scores[lr] = float(np.quantile([r['lexc'] for r in res], .75))
            pd.DataFrame([dict(arm=arm, family=fam, E=E, lr=lr, q75=scores[lr])]).to_csv(PT, mode='a', header=not PT.exists(), index=False)
        lr_star, pl = plateau_centre({l: scores[l] for l in grid}); LRSTAR[(arm, fam, E)] = lr_star
        if (arm, fam, E) in done: continue
        if fam in REAL: de, se = [REAL[fam](E)] * N_RUNS, [init_seed(arm, fam, E, t) for t in range(N_RUNS)]
        else: de, se = [synthetic(fam, E, t) for t in range(N_RUNS)], [init_seed(arm, fam, E, t) for t in range(N_RUNS)]
        res = run_cell(arm, E, de, se, lr_star, 'adam')
        pd.DataFrame([dict(arm=arm, family=fam, E=E, run=r, mode='adam', lr=lr_star, plateau_lo=pl[0], plateau_hi=pl[-1],
                           plateau_at_edge=bool(pl[0] == min(grid) or pl[-1] == max(grid)), **x) for r, x in enumerate(res)]).to_csv(PE, mode='a', header=not PE.exists(), index=False)
        lx = [x['lexc'] for x in res]
        print(f'{arm:9s} {fam:6s} E={E:2d} lr*={lr_star:g}: lexc median {np.median(lx):6.2f} [IQR {np.quantile(lx, .25):.2f},{np.quantile(lx, .75):.2f}] ({time.time() - t0:.0f}s)')

cells already evaluated (skipped): 48 | cells to evaluate: []


In [4]:
# ── Cell 2b: learning-rate robustness (appendix) — bracketing of the grid ────────────────────────────────────────
# Primary results (tuning.csv, eval.csv) are not touched. One pre-specified rule, applied to every architecture alike
# (the rule of NB1, cell 4b): if the near-best plateau of a configuration contains the smallest rate of its grid, the two
# next rates below are added to its tuning (1-3-10 steps); if it contains the largest, the next rate above; reselect
# with plateau_centre on the same tuning runs; if the selection changes, the new rate is evaluated on the same runs
# -> eval_ext.csv. One extension only; a plateau that still touches the extended grid is flagged (still_at_edge).
# SKIP_SATURATED: a configuration whose best tuning score is at the floor (every tuning run at the optimum up to the
# floor of lexc) is not extended, because no rate can score better there; it is flagged (saturated). Set it to False to
# apply the rule of NB1 without this exception. Run after cell 2; resumable.
SKIP_SATURATED, FLOOR_LEXC = True, -16.0
LADDER = [1e-5, 3e-5, 1e-4, 3e-4, 1e-3, 3e-3, 1e-2, 3e-2, 1e-1, 3e-1, 1.0]
def bracket(grid, pl):
    """rates to add: two ladder steps below the grid if the plateau contains its minimum, one above if its maximum"""
    lo, hi = LADDER.index(min(grid)), LADDER.index(max(grid))
    return (LADDER[max(lo - 2, 0):lo][::-1] if min(grid) in pl else []) + (LADDER[hi + 1:hi + 2] if max(grid) in pl else [])
def tune_and_eval_sets(arm, fam, E):
    """the tuning runs and the evaluation runs of cell 2 for this configuration (same data, same initial gates)"""
    real = fam in REAL; mk = (lambda t: REAL[fam](E)) if real else (lambda t: synthetic(fam, E, t))
    return ([mk(5000 + j) for j in range(N_TUNE)], [init_seed(arm, fam, E, 5000 + j) for j in range(N_TUNE)],
            [mk(t) for t in range(N_RUNS)], [init_seed(arm, fam, E, t) for t in range(N_RUNS)])
PX, PXT, PXS = OUT / 'eval_ext.csv', OUT / 'tuning_ext.csv', OUT / 'ext_selection.csv'
EV, TU = pd.read_csv(PE), pd.read_csv(PT)
def keys(path): return set(map(tuple, pd.read_csv(path)[['arm', 'family', 'E']].values)) if path.exists() else set()
def append(path, rows): pd.DataFrame(rows).to_csv(path, mode='a', header=not path.exists(), index=False)
doneS = keys(PXS); cfgs = [tuple(x) for x in EV[['arm', 'family', 'E', 'lr']].drop_duplicates().itertuples(index=False)]
print(f'configurations in eval.csv: {len(cfgs)} | bracketing done: {len(doneS)}')
for arm, fam, E, lr_star in cfgs:
    if (arm, fam, E) in doneS: continue
    if fam not in REAL and fam not in FAMILIES: print(f'!! {fam}: data not available, skipped'); continue
    t0 = time.time(); grid = LRS_REAL if fam in SCREENED else LRS; sel = (TU.arm == arm) & (TU.family == fam) & (TU.E == E)
    scores = {l: v for l, v in zip(TU[sel].lr, TU[sel].q75) if l in grid}; _, pl = plateau_centre(scores)
    saturated = bool(min(scores.values()) <= FLOOR_LEXC + 1e-9); extra = [] if (saturated and SKIP_SATURATED) else bracket(grid, pl)
    if PXT.exists():                                                             # extension already tuned (resume)
        T2 = pd.read_csv(PXT); s2 = (T2.arm == arm) & (T2.family == fam) & (T2.E == E); scores.update(zip(T2[s2].lr, T2[s2].q75))
    todo_lrs = [l for l in extra if l not in scores]; dt = st = de = se = None
    if todo_lrs:
        dt, st, de, se = tune_and_eval_sets(arm, fam, E)
        for lr in todo_lrs:
            res = run_cell(arm, E, dt, st, lr, 'adam'); scores[lr] = float(np.quantile([r['lexc'] for r in res], .75))
            append(PXT, [dict(arm=arm, family=fam, E=E, lr=lr, q75=scores[lr])])
    scores = {l: v for l, v in scores.items() if l in grid or l in extra}
    lr_new, pl_new = plateau_centre(scores); g = sorted(scores); changed = not np.isclose(lr_new, lr_star)
    if changed:
        if de is None: dt, st, de, se = tune_and_eval_sets(arm, fam, E)
        res = run_cell(arm, E, de, se, lr_new, 'adam')
        append(PX, [dict(arm=arm, family=fam, E=E, run=r, mode='adam', lr=lr_new, **x) for r, x in enumerate(res)])
    append(PXS, [dict(arm=arm, family=fam, E=E, lr_tuned=lr_star, lr_bracketed=lr_new, extended=' '.join(f'{l:g}' for l in extra),
                      plateau_lo=pl_new[0], plateau_hi=pl_new[-1], saturated=saturated,
                      still_at_edge=bool(pl_new[0] == g[0] or pl_new[-1] == g[-1]), rerun=changed)])
    print(f'{arm:9s} {fam:8s} E={E:2d}: lr {lr_star:g} -> {lr_new:g}' + (' (evaluated)' if changed else ' (unchanged)')
          + (' | saturated, not extended' if saturated and SKIP_SATURATED else
             (f' | extended by [{" ".join(f"{l:g}" for l in extra)}]' if extra else ' | plateau inside the grid, nothing to extend')) + f' ({time.time() - t0:.0f}s)')

# summary: every configuration under its bracketed rate (its primary runs where the rate did not change)
SEL = pd.read_csv(PXS); XE = pd.read_csv(PX) if PX.exists() else EV.iloc[:0]; parts = []
for s in SEL.itertuples():
    src = XE if s.rerun else EV; parts.append(src[(src.arm == s.arm) & (src.family == s.family) & (src.E == s.E)][['arm', 'family', 'E', 'run', 'lr', 'lexc', 'test_R2']])
BR = pd.concat(parts, ignore_index=True) if parts else EV.iloc[:0]; BR.to_csv(OUT / 'eval_bracketed.csv', index=False); rows = []
for s in SEL.itertuples():
    p = EV[(EV.arm == s.arm) & (EV.family == s.family) & (EV.E == s.E)]; b = BR[(BR.arm == s.arm) & (BR.family == s.family) & (BR.E == s.E)]
    rows.append(dict(family=s.family, E=s.E, arm=s.arm, lr_tuned=s.lr_tuned, lr_bracketed=s.lr_bracketed, saturated=s.saturated, still_at_edge=s.still_at_edge,
                     med_lexc_tuned=float(p.lexc.median()), med_lexc_bracketed=float(b.lexc.median()),
                     test_R2_tuned=float(p.test_R2.median()), test_R2_bracketed=float(b.test_R2.median())))
SX = pd.DataFrame(rows); SX.to_csv(OUT / 'summary_ext.csv', index=False)
if len(SX):
    ch = SX[~np.isclose(SX.lr_tuned, SX.lr_bracketed)]
    print(f'\nselection changed in {len(ch)} of {len(SX)} configurations; saturated ({'not extended' if SKIP_SATURATED else 'extended all the same'}): {int(SX.saturated.sum())}; still at the edge after the extension: '
          f'{list(SX[SX.still_at_edge & ~SX.saturated][["family", "E", "arm"]].itertuples(index=False, name=None))}')
    if len(ch): print(ch.sort_values(['family', 'E', 'arm']).round(3).to_string(index=False))
rows = []
for (fam, E), g in BR.groupby(['family', 'E']):
    s, a = g[g.arm == 'serial'].lexc.values, g[g.arm == 'ancilla'].lexc.values; p = EV[(EV.family == fam) & (EV.E == E)]
    if len(s) and len(a):
        rows.append(dict(family=fam, E=E, **dict(zip(['shift', 'lo', 'hi'], hl_shift(s, a))),
                         **dict(zip(['shift_tuned', 'lo_tuned', 'hi_tuned'], hl_shift(p[p.arm == 'serial'].lexc.values, p[p.arm == 'ancilla'].lexc.values)))))
SHX = pd.DataFrame(rows); SHX.to_csv(OUT / 'shift_serial_minus_ancilla_ext.csv', index=False)
if len(SHX):
    print('\nHodges-Lehmann shift of lexc, serial − ancilla (decades) [95%]: bracketed rates | tuned rates (primary)')
    print(SHX.assign(bracketed=SHX.apply(lambda r: f"{r['shift']:+.2f} [{r.lo:+.2f},{r.hi:+.2f}]", axis=1),
                     tuned=SHX.apply(lambda r: f'{r.shift_tuned:+.2f} [{r.lo_tuned:+.2f},{r.hi_tuned:+.2f}]', axis=1))[['family', 'E', 'bracketed', 'tuned']].to_string(index=False))

configurations in eval.csv: 48 | bracketing done: 48

selection changed in 4 of 48 configurations; saturated (not extended): 30; still at the edge after the extension: [('nottem', 24, 'ancilla'), ('nottem', 32, 'serial'), ('gasprice', 24, 'serial')]
  family  E     arm  lr_tuned  lr_bracketed  saturated  still_at_edge  med_lexc_tuned  med_lexc_bracketed  test_R2_tuned  test_R2_bracketed
    bump 32  serial     0.003         0.001      False          False          -2.365              -2.373          0.986              0.985
gasprice 24  serial     0.003         0.010      False           True          -2.702              -2.692          0.957              0.954
  nottem 32 ancilla     0.010         0.003      False          False         -15.969             -15.969          0.887              0.887
  nottem 32  serial     0.003         0.001      False           True          -2.252              -2.259          0.888              0.888

Hodges-Lehmann shift of lexc, serial − ancilla (d

In [5]:
# ── Cell 2c: step budget (appendix) — does the ordering survive BUDGET_FACTOR times as many Adam steps? ──────────
# Primary results are not touched. Pre-specified subset: serial and ancilla; synthetic families at E in ATTR_E, real
# series at the smaller E of E_REAL; the first BUDGET_RUNS evaluation runs of cell 2 (same data, same initial gates);
# BUDGET_FACTOR x STEPS steps at the learning rate selected in cell 2. The rate is not re-tuned for the longer run; the
# cosine schedule is stretched to it. -> budget.csv. Compared with the same runs at STEPS steps from eval.csv. Resumable.
# Caution (found after the first full run): stretching also slows the final decay, and Adam then leaves converged runs at
# lexc of about -7 to -8 instead of -16. Medians, shares at the floor and shifts in decades of this cell are therefore not
# comparable between the two budgets; use the success rates of cell 2d.
BUDGET_FACTOR, BUDGET_RUNS = 4, min(N_RUNS, 30)
PB = OUT / 'budget.csv'; EV = pd.read_csv(PE)
doneB = set(map(tuple, pd.read_csv(PB)[['arm', 'family', 'E']].values)) if PB.exists() else set()
bcells = [(fam, E) for fam in FAMILIES for E in ATTR_E if E in E_LIST] + [(fam, E_REAL[0]) for fam in REAL]
print(f'budget ablation: {BUDGET_FACTOR} x {STEPS} steps, {BUDGET_RUNS} runs, configurations {bcells} x {ATTR_ARMS}')
for fam, E in bcells:
    for arm in ATTR_ARMS:
        ev = EV[(EV.arm == arm) & (EV.family == fam) & (EV.E == E)]
        if (arm, fam, E) in doneB or not len(ev): continue
        t0 = time.time(); lr_star = float(ev.lr.iloc[0]); mk = (lambda t: REAL[fam](E)) if fam in REAL else (lambda t: synthetic(fam, E, t))
        de, se = [mk(t) for t in range(BUDGET_RUNS)], [init_seed(arm, fam, E, t) for t in range(BUDGET_RUNS)]
        res = run_cell(arm, E, de, se, lr_star, 'adam', steps=BUDGET_FACTOR * STEPS)
        pd.DataFrame([dict(arm=arm, family=fam, E=E, run=r, steps=BUDGET_FACTOR * STEPS, lr=lr_star, **x) for r, x in enumerate(res)]).to_csv(PB, mode='a', header=not PB.exists(), index=False)
        print(f'{arm:9s} {fam:8s} E={E:2d} lr*={lr_star:g}: lexc median {np.median([x["lexc"] for x in res]):6.2f} ({time.time() - t0:.0f}s)')
if PB.exists():
    B = pd.read_csv(PB); rows = []
    for (fam, E), b in B.groupby(['family', 'E']):
        e = EV[(EV.family == fam) & (EV.E == E) & (EV.run < BUDGET_RUNS)]; row = dict(family=fam, E=E, runs=int(b.run.nunique()))
        for arm in ATTR_ARMS:
            x1, x4 = e[e.arm == arm].set_index('run').lexc, b[b.arm == arm].set_index('run').lexc; d = (x4 - x1).dropna()
            row.update({f'{arm}_med_1x': float(x1.median()), f'{arm}_med_{BUDGET_FACTOR}x': float(x4.median()),
                        f'{arm}_paired_gain': float(d.median()) if len(d) else np.nan,                       # median over runs of lexc(long) − lexc(short); negative = better
                        f'{arm}_floor_1x': float((x1 <= -15).mean()), f'{arm}_floor_{BUDGET_FACTOR}x': float((x4 <= -15).mean())})
        for tag, src in (('1x', e), (f'{BUDGET_FACTOR}x', b)):
            s, a = src[src.arm == 'serial'].lexc.values, src[src.arm == 'ancilla'].lexc.values
            if len(s) and len(a): row.update(dict(zip([f'shift_{tag}', f'lo_{tag}', f'hi_{tag}'], hl_shift(s, a))))
        rows.append(row)
    BS = pd.DataFrame(rows); BS.to_csv(OUT / 'budget_summary.csv', index=False); f4 = f'{BUDGET_FACTOR}x'
    print(f'\nmedian lexc at {STEPS} and at {BUDGET_FACTOR * STEPS} steps (same runs); gain = median over runs of the paired difference (negative: the longer run is better);')
    print('share of runs at the optimum (lexc <= -15); Hodges-Lehmann shift serial − ancilla [95%]')
    fm = lambda r, t: f"{r[f'shift_{t}']:+.2f} [{r[f'lo_{t}']:+.2f},{r[f'hi_{t}']:+.2f}]" if f'shift_{t}' in r and np.isfinite(r[f'shift_{t}']) else ''
    show = BS.assign(shift_1x_=BS.apply(lambda r: fm(r, '1x'), axis=1), shift_long=BS.apply(lambda r: fm(r, f4), axis=1))
    cols = ['family', 'E', 'runs'] + [f'{a}_{k}' for a in ATTR_ARMS for k in ('med_1x', f'med_{f4}', 'paired_gain', 'floor_1x', f'floor_{f4}')] + ['shift_1x_', 'shift_long']
    print(show[cols].round(2).to_string(index=False))

budget ablation: 4 x 20000 steps, 30 runs, configurations [('flat', 12), ('flat', 24), ('bump', 12), ('bump', 24), ('nottem', 24), ('gasprice', 24), ('ibmclose', 24)] x ['serial', 'ancilla']

median lexc at 20000 and at 80000 steps (same runs); gain = median over runs of the paired difference (negative: the longer run is better);
share of runs at the optimum (lexc <= -15); Hodges-Lehmann shift serial − ancilla [95%]
  family  E  runs  serial_med_1x  serial_med_4x  serial_paired_gain  serial_floor_1x  serial_floor_4x  ancilla_med_1x  ancilla_med_4x  ancilla_paired_gain  ancilla_floor_1x  ancilla_floor_4x              shift_1x_          shift_long
    bump 12    30          -4.26          -7.06               -0.00             0.27              0.0          -16.00           -7.55                 8.45              1.00               0.0  +11.48 [+2.46,+12.36] +0.33 [-1.48,+1.40]
    bump 24    30          -2.78          -3.35               -0.60             0.00              0.0           

In [6]:
# ── Cell 2d: step budget with the final decay unchanged (appendix), and success rates for all schedules ─────────
# Cell 2c stretches the cosine schedule over BUDGET_FACTOR x STEPS steps. That also slows the final decay of the rate,
# and with the slower decay Adam ends converged runs at lexc of about -7 to -8 instead of -16 (every ancilla run in
# budget.csv). So `lexc <= -15` and shifts in decades are not comparable between cell 2 and cell 2c: they measure the
# end of the schedule, not the circuit. Two consequences, both handled here:
#  (1) budget with the end of the run unchanged: the rate is held at its initial value for (BUDGET_FACTOR - 1) x STEPS
#      steps and then follows the cosine schedule of cell 2 over the last STEPS steps. Same configurations, runs and
#      rates as cell 2c. -> budget_hold.csv. Set RUN_HOLD = False to skip the runs and only compute the summary.
#  (2) a statistic that does not depend on the end of the schedule: runs with lexc <= thr, for thr in BUDGET_THR, with
#      the ancilla − serial difference [Newcombe 95%], for STEPS steps, the stretched schedule (2c) and the held one.
#      -> budget_rates.csv. Under the stretched schedule the level at which Adam leaves a converged run depends on the
#      rate (roughly the square of the final rate: about -7 to -10 for lr <= 0.01, up to -5 for lr = 0.03), so no single
#      threshold separates converged from failed runs for every rate there. The rates of the stretched schedule are
#      indicative (thr = -6 is strict, -4 lenient; -15 is empty there); 1x against the held schedule is comparable at -15.
# The two long schedules ask different things: the stretched one, whether a slower decay helps; the held one, whether
# more steps at the initial rate help.
from phaselock_core import newcombe
RUN_HOLD, BUDGET_THR = True, (-15, -6, -4)
BUDGET_FACTOR, BUDGET_RUNS = 4, min(N_RUNS, 30)                                  # as in cell 2c
PB, PBH = OUT / 'budget.csv', OUT / 'budget_hold.csv'; EV = pd.read_csv(PE)
doneH = set(map(tuple, pd.read_csv(PBH)[['arm', 'family', 'E']].values)) if PBH.exists() else set()
bcells = [(fam, E) for fam in FAMILIES for E in ATTR_E if E in E_LIST] + [(fam, E_REAL[0]) for fam in REAL]
if RUN_HOLD:
    print(f'held schedule: {(BUDGET_FACTOR - 1) * STEPS} steps at the initial rate, then the decay of cell 2 over {STEPS} steps; {BUDGET_RUNS} runs')
    for fam, E in bcells:
        for arm in ATTR_ARMS:
            ev = EV[(EV.arm == arm) & (EV.family == fam) & (EV.E == E)]
            if (arm, fam, E) in doneH or not len(ev): continue
            t0 = time.time(); lr_star = float(ev.lr.iloc[0]); mk = (lambda t: REAL[fam](E)) if fam in REAL else (lambda t: synthetic(fam, E, t))
            de, se = [mk(t) for t in range(BUDGET_RUNS)], [init_seed(arm, fam, E, t) for t in range(BUDGET_RUNS)]
            res = run_cell(arm, E, de, se, lr_star, 'adam', steps=BUDGET_FACTOR * STEPS, hold=(BUDGET_FACTOR - 1) * STEPS)
            pd.DataFrame([dict(arm=arm, family=fam, E=E, run=r, steps=BUDGET_FACTOR * STEPS, hold=(BUDGET_FACTOR - 1) * STEPS, lr=lr_star, **x)
                          for r, x in enumerate(res)]).to_csv(PBH, mode='a', header=not PBH.exists(), index=False)
            print(f'{arm:9s} {fam:8s} E={E:2d} lr*={lr_star:g}: lexc median {np.median([x["lexc"] for x in res]):6.2f} ({time.time() - t0:.0f}s)')
SCHED = {'1x': EV[EV.run < BUDGET_RUNS]}
if PB.exists(): SCHED[f'{BUDGET_FACTOR}x stretched'] = pd.read_csv(PB)
if PBH.exists(): SCHED[f'{BUDGET_FACTOR}x held'] = pd.read_csv(PBH)
rows = []
for fam, E in bcells:
    for name, D in SCHED.items():
        d = D[(D.family == fam) & (D.E == E)]; s, a = d[d.arm == 'serial'].lexc, d[d.arm == 'ancilla'].lexc
        if not (len(s) and len(a)): continue
        for thr in BUDGET_THR:
            ks, ka = int((s <= thr).sum()), int((a <= thr).sum()); dd, lo, hi = newcombe(ka, len(a), ks, len(s))
            rows.append(dict(family=fam, E=E, schedule=name, thr=thr, k_serial=ks, n_serial=len(s), k_ancilla=ka, n_ancilla=len(a), diff=dd, lo=lo, hi=hi,
                             med_serial=float(s.median()), med_ancilla=float(a.median())))
BR_ = pd.DataFrame(rows); BR_.to_csv(OUT / 'budget_rates.csv', index=False)
if len(BR_):
    for thr in BUDGET_THR:
        b = BR_[BR_.thr == thr]
        print(f'\nruns with lexc <= {thr}: serial | ancilla, and ancilla − serial [Newcombe 95%]')
        print(b.assign(cell=b.apply(lambda r: f'{r.k_serial:2d}/{r.n_serial} | {r.k_ancilla:2d}/{r.n_ancilla}  {r["diff"]:+.2f} [{r.lo:+.2f},{r.hi:+.2f}]', axis=1))
               .pivot_table(index=['family', 'E'], columns='schedule', values='cell', aggfunc='first')[list(SCHED)].to_string())
    print('\nmedian lexc, serial | ancilla (the ancilla column shows where each schedule leaves converged runs)')
    b = BR_[BR_.thr == BUDGET_THR[0]]
    print(b.assign(cell=b.apply(lambda r: f'{r.med_serial:6.1f} | {r.med_ancilla:6.1f}', axis=1)).pivot_table(index=['family', 'E'], columns='schedule', values='cell', aggfunc='first')[list(SCHED)].to_string())

held schedule: 60000 steps at the initial rate, then the decay of cell 2 over 20000 steps; 30 runs

runs with lexc <= -15: serial | ancilla, and ancilla − serial [Newcombe 95%]
schedule                                     1x                        4x stretched                             4x held
family   E                                                                                                             
bump     12   8/30 | 30/30  +0.73 [+0.52,+0.86]   0/30 |  0/30  +0.00 [-0.11,+0.11]  15/30 | 30/30  +0.50 [+0.30,+0.67]
         24   0/30 |  7/30  +0.23 [+0.07,+0.41]   0/30 |  0/30  +0.00 [-0.11,+0.11]   0/30 | 22/30  +0.73 [+0.52,+0.86]
flat     12  23/30 | 30/30  +0.23 [+0.07,+0.41]   0/30 |  0/30  +0.00 [-0.11,+0.11]  24/30 | 30/30  +0.20 [+0.05,+0.37]
         24  20/30 | 30/30  +0.33 [+0.15,+0.51]   0/30 |  0/30  +0.00 [-0.11,+0.11]  14/30 | 30/30  +0.53 [+0.33,+0.70]
gasprice 24   0/30 | 30/30  +1.00 [+0.84,+1.00]   0/30 |  0/30  +0.00 [-0.11,+0.11]   1/30 | 30/30  +0.

In [7]:
# ── Cell 3: reference — Levenberg-Marquardt on the same runs (same data and initial gates as cell 2) ─────────────
# LM_SCOPE = 'all': every configuration of cell 2. Resumable: a configuration is written when it is finished, and the
# configurations already in attribution.csv are skipped (the eight of the earlier subset are kept as they are).
# Order: ascending E, synthetic before real, so that an interrupted run leaves the cheap configurations complete.
# Time per configuration (100 runs, 2,000 iterations) on the laptop CPU, from the earlier run: 40-55 min at E = 12,
# 2.2-3.8 h at E = 24; expect about half of that at E = 8 and about 1.7 times as much at E = 32.
PA = OUT / 'attribution.csv'
doneA = set(map(tuple, pd.read_csv(PA)[['arm', 'family', 'E', 'mode']].values)) if PA.exists() else set()
EV = pd.read_csv(PE)
cells_all = [(fam, E) for fam in FAMILIES for E in E_LIST] + [(fam, E) for fam in REAL for E in E_REAL]       # as in cell 2
if LM_SCOPE == 'all': lm_cells = [(fam, E, arm) for fam, E in cells_all for arm in ARM_LIST]
else: lm_cells = [(fam, E, arm) for fam in FAMILIES for E in ATTR_E if E in E_LIST for arm in ATTR_ARMS]
lm_cells = sorted([c for c in lm_cells if not c[1] % ARMS[c[2]]['div']], key=lambda c: (c[1], c[0] in REAL, c[0], ARM_LIST.index(c[2])))
todo = [c for c in lm_cells if (c[2], c[0], c[1], 'lm') not in doneA]
print(f'Levenberg-Marquardt ({LM_SCOPE}): {len(lm_cells)} configurations, {len(lm_cells) - len(todo)} already in {PA.name}; to run: {todo}')
for fam, E, arm in todo:
    sel = EV[(EV.arm == arm) & (EV.family == fam) & (EV.E == E)]
    if not len(sel): print(f'!! {arm} {fam} E={E}: not in eval.csv (run cell 2 first); skipped'); continue
    de = [REAL[fam](E)] * N_RUNS if fam in REAL else [synthetic(fam, E, t) for t in range(N_RUNS)]             # data of cell 2
    se = [init_seed(arm, fam, E, t) for t in range(N_RUNS)]                                                    # initial gates of cell 2
    lr_star = float(sel.lr.iloc[0])                                    # written to the file for reference; Levenberg-Marquardt does not use it
    t0 = time.time(); res = run_cell(arm, E, de, se, lr_star, 'lm')
    pd.DataFrame([dict(arm=arm, family=fam, E=E, run=r, mode='lm', lr=lr_star, **x) for r, x in enumerate(res)]).to_csv(PA, mode='a', header=not PA.exists(), index=False)
    lx = np.array([x['lexc'] for x in res])
    print(f'lm {arm:9s} {fam:8s} E={E:2d}: lexc median {np.median(lx):6.2f}, at the optimum (lexc <= -6) {int((lx <= -6).sum())} of {len(lx)} ({time.time() - t0:.0f}s)')


Levenberg-Marquardt (all): 48 configurations, 8 already in attribution.csv; to run: [('bump', 8, 'serial'), ('bump', 8, 'ancilla'), ('bump', 8, 'parallel2'), ('flat', 8, 'serial'), ('flat', 8, 'ancilla'), ('flat', 8, 'parallel2'), ('bump', 12, 'parallel2'), ('flat', 12, 'parallel2'), ('bump', 16, 'serial'), ('bump', 16, 'ancilla'), ('bump', 16, 'parallel2'), ('flat', 16, 'serial'), ('flat', 16, 'ancilla'), ('flat', 16, 'parallel2'), ('bump', 24, 'parallel2'), ('flat', 24, 'parallel2'), ('gasprice', 24, 'serial'), ('gasprice', 24, 'ancilla'), ('gasprice', 24, 'parallel2'), ('ibmclose', 24, 'serial'), ('ibmclose', 24, 'ancilla'), ('ibmclose', 24, 'parallel2'), ('nottem', 24, 'serial'), ('nottem', 24, 'ancilla'), ('nottem', 24, 'parallel2'), ('bump', 32, 'serial'), ('bump', 32, 'ancilla'), ('bump', 32, 'parallel2'), ('flat', 32, 'serial'), ('flat', 32, 'ancilla'), ('flat', 32, 'parallel2'), ('gasprice', 32, 'serial'), ('gasprice', 32, 'ancilla'), ('gasprice', 32, 'parallel2'), ('ibmclose'

In [8]:
# ── Cell 4: summary tables (the same quantities are recomputed by figures.ipynb) ──
EV = pd.read_csv(PE); AT = pd.read_csv(PA) if PA.exists() else pd.DataFrame()
q = lambda p: (lambda v: np.quantile(v, p))
S = EV.groupby(['family', 'E', 'arm']).agg(med=('lexc', 'median'), q25=('lexc', q(.25)), q75=('lexc', q(.75)), testR2=('test_R2', 'median'),
                                           lr=('lr', 'first'), edge=('plateau_at_edge', 'first'), n=('lexc', 'size')).reset_index()
S.to_csv(OUT / 'summary.csv', index=False)
print('median lexc [IQR] per arm (and median test R2):')
print(S.assign(cell=S.apply(lambda r: f"{r.med:6.2f} [{r.q25:.1f},{r.q75:.1f}] R2 {r.testR2:.3f}", axis=1)).pivot_table(index=['family', 'E'], columns='arm', values='cell', aggfunc='first'))
rows = []
for (fam, E), g in EV.groupby(['family', 'E']):
    s, a = g[g.arm == 'serial'].lexc.values, g[g.arm == 'ancilla'].lexc.values
    if len(s) and len(a): rows.append(dict(family=fam, E=E, mode='adam', **dict(zip(['shift', 'lo', 'hi'], hl_shift(s, a)))))
if len(AT):
    for (fam, E, mode), g in AT.groupby(['family', 'E', 'mode']):
        s, a = g[g.arm == 'serial'].lexc.values, g[g.arm == 'ancilla'].lexc.values
        if len(s) and len(a): rows.append(dict(family=fam, E=E, mode=mode, **dict(zip(['shift', 'lo', 'hi'], hl_shift(s, a)))))
SH = pd.DataFrame(rows); SH.to_csv(OUT / 'shift_serial_minus_ancilla.csv', index=False)
print('\nHodges-Lehmann shift of lexc, serial − ancilla (decades), 95% bootstrap CI; mode: adam or lm:')
print(SH.assign(cell=SH.apply(lambda r: f"{r['shift']:+.2f} [{r.lo:+.2f},{r.hi:+.2f}]", axis=1)).pivot_table(index=['family', 'mode'], columns='E', values='cell', aggfunc='first'))
if S.edge.any(): print('\nnote: near-best learning-rate plateau touches the grid edge for', list(S[S.edge][['family', 'E', 'arm']].itertuples(index=False, name=None)))

median lexc [IQR] per arm (and median test R2):
arm                                ancilla                      parallel2                         serial
family   E                                                                                              
bump     8   -16.00 [-16.0,-16.0] R2 0.997  -16.00 [-16.0,-16.0] R2 0.997  -16.00 [-16.0,-10.1] R2 0.997
         12  -16.00 [-16.0,-16.0] R2 0.997  -16.00 [-16.0,-16.0] R2 0.997    -4.51 [-16.0,-3.2] R2 0.996
         16  -16.00 [-16.0,-16.0] R2 0.997  -16.00 [-16.0,-16.0] R2 0.997     -3.25 [-4.1,-2.8] R2 0.995
         24    -8.92 [-16.0,-4.3] R2 0.995    -4.15 [-15.8,-3.4] R2 0.995     -2.72 [-3.1,-2.4] R2 0.992
         32     -3.55 [-5.0,-3.0] R2 0.993     -3.36 [-4.2,-2.9] R2 0.992     -2.36 [-2.9,-2.1] R2 0.986
flat     8   -16.00 [-16.0,-16.0] R2 1.000  -16.00 [-16.0,-16.0] R2 1.000  -16.00 [-16.0,-16.0] R2 1.000
         12  -16.00 [-16.0,-16.0] R2 1.000  -16.00 [-16.0,-16.0] R2 1.000  -16.00 [-16.0,-16.0] R2 1.000
       